# Bronze Layer

In [0]:
volume_path = "/Volumes/parcels_project/bronze/raw_dataset/"

try:
    loaded_files = [
        row.source_file for row in
        spark.sql("SELECT source_file FROM parcels_project.bronze.property_raw").collect()
    ]
except Exception:
    loaded_files = []


all_files = [f.name for f in dbutils.fs.ls(volume_path) if f.name.lower().endswith(".csv")]

new_files = [f for f in all_files if f not in loaded_files]

if not new_files:
    print("No new files found — bronze layer is already up to date.")
else:
    print(f"Found {len(new_files)} new file(s) to load: {new_files}")


No new files found — bronze layer is already up to date.


In [0]:
pre_load_count = spark.sql(
    "SELECT COUNT(*) AS cnt FROM parcels_project.bronze.property_raw"
).collect()[0]["cnt"]

print(f"Row count before load: {pre_load_count:,}")

Row count before load: 437,548


In [0]:
raw_file_counts = {}

for file_name in new_files:
    raw_count = spark.read.csv(volume_path + file_name, header=True, inferSchema=True).count()
    raw_file_counts[file_name] = raw_count
    print(f"{file_name}: {raw_count:,} rows in source")

In [0]:
from pyspark.sql.functions import current_timestamp, lit

total_rows = 0

for file_name in new_files:
    df = spark.read.csv(volume_path + file_name, header=True, inferSchema=True)

    df_bronze = (df.withColumn("ingest_timestamp", current_timestamp())
                   .withColumn("source_file", lit(file_name)))

    n = df_bronze.count()
    total_rows += n
    print(f"{file_name}: {n} rows ingested")

    df_bronze.write.format("delta").mode("append").saveAsTable("parcels_project.bronze.property_raw")

print(f"\nTotal rows appended: {total_rows}")


Total rows appended: 0


In [0]:
mismatch_found = False

for file_name, raw_count in raw_file_counts.items():
    loaded_count = spark.sql(
        f"SELECT COUNT(*) AS cnt FROM parcels_project.bronze.property_raw WHERE source_file = '{file_name}'"
    ).collect()[0]["cnt"]

    status = "OK" if raw_count == loaded_count else "MISMATCH"
    if status == "MISMATCH":
        mismatch_found = True

    print(f"{file_name}: source={raw_count:,} | bronze={loaded_count:,} | {status}")

if mismatch_found:
    print("\nWarning: row counts do not match — investigate before proceeding to Silver.")

In [0]:
%sql
SELECT 
    COUNT(*) AS total_rows,
    MAX(ingest_timestamp) AS latest_load
FROM parcels_project.bronze.property_raw;

total_rows,latest_load
437548,2026-09-04T19:20:31.840Z


In [0]:
%sql

WITH load_history AS (
    SELECT 
        source_file,
        COUNT(*) AS records,

        MIN(ingest_timestamp) AS first_load,
        MAX(ingest_timestamp) AS last_load
    FROM parcels_project.bronze.property_raw
    GROUP BY source_file
),
data_quality AS (
    SELECT
        COUNT(*) AS total_records,
        COUNT(*) - COUNT(PARCEL_PK) AS null_parcel_pks,
        CASE 
            WHEN COUNT(*) - COUNT(PARCEL_PK) = 0 THEN 'No NULL PKs'
            ELSE 'Has NULL PKs'
        END AS quality_status
    FROM parcels_project.bronze.property_raw
)
SELECT 'Load History' AS section, * FROM load_history
UNION ALL
SELECT 'Data Quality' AS section, 
       CAST(total_records AS STRING) AS source_file,
       null_parcel_pks AS unique_parcels,
       NULL AS first_load,
       NULL AS last_load
FROM data_quality
ORDER BY section DESC, last_load DESC NULLS LAST;

section,source_file,records,first_load,last_load
Load History,new_parcel4.csv,17,2026-09-04T19:20:31.840Z,2026-09-04T19:20:31.840Z
Load History,new_parcel6.csv,3,2026-09-03T21:27:42.666Z,2026-09-03T21:27:42.666Z
Load History,Property.csv,437523,2026-09-03T21:25:51.832Z,2026-09-03T21:25:51.832Z
Load History,new_property1.csv,5,2026-09-03T20:13:09.964Z,2026-09-03T20:13:09.964Z
Data Quality,437548,120,null,null


In [0]:
%sql
SELECT source_file, COUNT(*) FROM parcels_project.bronze.property_raw
GROUP BY source_file;

source_file,COUNT(*)
Property.csv,437523
new_property1.csv,5
new_parcel4.csv,17
new_parcel6.csv,3
